# 实验 3 — 新增货币：schema 自动演化

**目标**：给 dlt pipeline 加一个新货币（AUD），重新 ingest，观察 DuckDB schema 是否自动更新。

**关键问题**：dlt 怎么做到不修改 SQL DDL 就能扩展列/行？

In [ ]:
import duckdb

DB = "../data/sandbox.duckdb"
con = duckdb.connect(DB)

print("=== 加 AUD 前 ===")
print("distinct currencies:", 
      con.execute("SELECT DISTINCT currency FROM raw_ecb.daily_rates ORDER BY 1").fetchall())
print("total rows:", con.execute("SELECT COUNT(*) FROM raw_ecb.daily_rates").fetchone()[0])

In [ ]:
# 手动操作：
# 1. 打开 dlt_pipelines/ecb_rates.py
# 2. 在 CURRENCIES 列表里加 "AUD"
# 3. 在 CURRENCY_NAMES 里加 "AUD": "Australian Dollar"
# 4. 执行下面这个 cell

import subprocess, os

result = subprocess.run(
    ["uv", "run", "python", "dlt_pipelines/ecb_rates.py"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
print("=== 加 AUD 后 ===")
con2 = duckdb.connect(DB)
print("distinct currencies:",
      con2.execute("SELECT DISTINCT currency FROM raw_ecb.daily_rates ORDER BY 1").fetchall())
print("total rows:", con2.execute("SELECT COUNT(*) FROM raw_ecb.daily_rates").fetchone()[0])
print("currencies_meta rows:", con2.execute("SELECT * FROM raw_ecb.currencies_meta").fetchall())

In [ ]:
# dlt 是怎么知道 schema 变了的？看 _dlt_version 表
con3 = duckdb.connect(DB)
con3.execute("SELECT * FROM raw_ecb._dlt_version ORDER BY inserted_at DESC LIMIT 3").df()

## 思考

- dlt 把 schema 存在 `_dlt_version` 表里，每次跑前和当前数据对比，自动做 ALTER TABLE / 插入新行
- `merge` disposition 用 `(date, currency)` 作 primary key，所以 AUD 的历史数据不会和旧数据冲突
- 如果你把 AUD 从 CURRENCIES 列表中删掉，再跑一次，AUD 行会消失吗？（不会——merge 不删行）